In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json

# Load all necessary data
df = pd.read_csv('../data/data_with_engineered_features.csv')
with open('../data/kpi_report.json', 'r') as f:
    kpi_report = json.load(f)

print("Data loaded successfully!")
print(f"Dashboard will be built using {len(df):,} customer records")

Data loaded successfully!
Dashboard will be built using 10,000 customer records


# Phase 5: Streamlit Dashboard Setup

## Interactive Business Intelligence Platform

This phase shows how to build the main DELIVERABLE: A Streamlit dashboard that brings all analysis to life.

### Dashboard Pages
1. **Overview Dashboard** - KPI summary and health metrics
2. **Engagement Analytics** - Deep-dive into activity patterns
3. **Product Utilization** - Product mix and cross-sell insights
4. **Premium Risk Detection** - At-risk wealthy customer identification
5. **Relationship Strength Scoring** - Customer loyalty tiers and stickiness

---

## Objective
Create an executive-ready, interactive dashboard for real-time decision-making

## Streamlit Dashboard Code


This code demonstrates dashboard structure and key visualizations.

In [6]:
# Create dashboard code template
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json

# Page configuration
st.set_page_config(page_title="Bank Churn Analytics", layout="wide", initial_sidebar_state="expanded")

# Custom CSS
st.markdown("""
<style>
    .metric-card { 
        background-color: #f0f2f6; 
        padding: 20px; 
        border-radius: 10px; 
        margin: 10px 0;
    }
    .title { color: #1f77b4; font-weight: bold; }
</style>
""", unsafe_allow_html=True)

# Load data
@st.cache_data
def load_data():
    df = pd.read_csv('data/data_with_engineered_features.csv')
    with open('data/kpi_report.json', 'r') as f:
        kpi = json.load(f)
    return df, kpi

df, kpi = load_data()

# Sidebar navigation
st.sidebar.title("📊 Bank Churn Dashboard")
page = st.sidebar.radio("Select Page", [
    "📈 Overview",
    "🎯 Engagement Analytics", 
    "📦 Product Utilization",
    "⚠️ Premium Risk Detection",
    "💎 Relationship Strength"
])

# ==================== PAGE 1: OVERVIEW ====================
if page == "📈 Overview":
    st.title("Overview Dashboard")
    
    # KPI Cards
    col1, col2, col3, col4 = st.columns(4)
    
    with col1:
        st.metric("Total Customers", f"{kpi['OVERALL_HEALTH']['Total_Customers']:,}")
        
    with col2:
        st.metric("Churned Customers", f"{kpi['OVERALL_HEALTH']['Churned_Customers']:,}", 
                 delta=f"{kpi['OVERALL_HEALTH']['Overall_Churn_Rate']}")
        
    with col3:
        st.metric("Retention Rate", kpi['OVERALL_HEALTH']['Overall_Retention_Rate'])
        
    with col4:
        st.metric("Active Members", kpi['ENGAGEMENT']['Active_Member_Ratio'])
    
    # Main visualizations
    col1, col2 = st.columns(2)
    
    with col1:
        # Churn distribution
        churn_data = df['Exited'].value_counts()
        fig_churn = go.Figure(data=[go.Pie(
            labels=['Retained', 'Churned'],
            values=[churn_data[0], churn_data[1]],
            marker_colors=['#2ecc71', '#e74c3c'],
            hole=0.3
        )])
        fig_churn.update_layout(title="Churn vs Retention")
        st.plotly_chart(fig_churn, use_container_width=True)
    
    with col2:
        # Customer value distribution
        value_dist = df['Customer_Value_Score'].value_counts().sort_index()
        fig_value = go.Figure(data=[go.Bar(x=value_dist.index, y=value_dist.values, 
                                          marker_color='#3498db')])
        fig_value.update_layout(title="Customer Value Distribution", 
                               xaxis_title="Value Score", yaxis_title="Count")
        st.plotly_chart(fig_value, use_container_width=True)
    
    # Key metrics table
    st.subheader("Key Metrics Summary")
    metrics_table = pd.DataFrame({
        'Metric': list(kpi['ENGAGEMENT'].keys()),
        'Value': list(kpi['ENGAGEMENT'].values())
    })
    st.dataframe(metrics_table, use_container_width=True)

# ==================== PAGE 2: ENGAGEMENT ====================
elif page == "🎯 Engagement Analytics":
    st.title("Engagement Analytics")
    
    # Engagement vs Churn
    engagement_churn = df.groupby('IsActiveMember')['Exited'].agg(['count', 'mean'])
    
    col1, col2 = st.columns(2)
    with col1:
        fig = go.Figure()
        fig.add_trace(go.Bar(x=['Inactive', 'Active'], 
                            y=[engagement_churn.loc[0, 'count'], engagement_churn.loc[1, 'count']],
                            marker_color=['#e74c3c', '#2ecc71']))
        fig.update_layout(title="Customers by Engagement Status")
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        fig = go.Figure()
        fig.add_trace(go.Bar(x=['Inactive', 'Active'],
                            y=[engagement_churn.loc[0, 'mean']*100, engagement_churn.loc[1, 'mean']*100],
                            marker_color=['#e74c3c', '#2ecc71']))
        fig.update_layout(title="Churn Rate by Engagement", yaxis_title="Churn Rate (%)")
        st.plotly_chart(fig, use_container_width=True)

# ==================== PAGE 3: PRODUCTS ====================
elif page == "📦 Product Utilization":
    st.title("Product Utilization Analysis")
    
    product_analysis = df.groupby('NumOfProducts')['Exited'].agg(['count', 'mean'])
    
    col1, col2 = st.columns(2)
    with col1:
        fig = go.Figure()
        fig.add_trace(go.Bar(x=product_analysis.index, y=product_analysis['count'],
                            marker_color='#3498db'))
        fig.update_layout(title="Customers by Product Count", xaxis_title="Products", yaxis_title="Count")
        st.plotly_chart(fig, use_container_width=True)
    
    with col2:
        fig = go.Figure()
        fig.add_trace(go.Bar(x=product_analysis.index, y=product_analysis['mean']*100,
                            marker_color='#e74c3c'))
        fig.update_layout(title="Churn by Product Count", xaxis_title="Products", yaxis_title="Churn %")
        st.plotly_chart(fig, use_container_width=True)

# ==================== PAGE 4: PREMIUM RISK ====================
elif page == "⚠️ Premium Risk Detection":
    st.title("Premium Customer Risk Analysis")
    
    # Filters
    col1, col2 = st.columns(2)
    with col1:
        balance_threshold = st.slider("Balance Threshold", 0, 200000, 100000, 10000)
    with col2:
        salary_threshold = st.slider("Salary Threshold", 0, 200000, 150000, 10000)
    
    # Find at-risk premium customers
    at_risk = df[(df['Balance'] > balance_threshold) & 
                 (df['EstimatedSalary'] > salary_threshold) & 
                 (df['IsActiveMember'] == 0)]
    
    st.metric("At-Risk Premium Customers", len(at_risk))
    st.metric("Total Balance at Risk", f"${at_risk['Balance'].sum():,.0f}")
    st.metric("Churn Rate", f"{at_risk['Exited'].mean():.2%}")
    
    # Display customer list
    st.subheader("At-Risk Customers")
    display_cols = ['CustomerId', 'Age', 'Geography', 'Balance', 'EstimatedSalary', 'Exited', 'Tenure']
    st.dataframe(at_risk[display_cols].sort_values('Balance', ascending=False))

# ==================== PAGE 5: LOYALTY ====================
else:  # Relationship Strength
    st.title("Relationship Strength Scoring")
    
    # Sticky customers
    sticky_count = df['Sticky_Customer'].sum()
    sticky_churn = df[df['Sticky_Customer'] == 1]['Exited'].mean()
    
    col1, col2, col3 = st.columns(3)
    with col1:
        st.metric("Sticky Customers", f"{sticky_count:,}")
    with col2:
        st.metric("Sticky Customer %", f"{sticky_count/len(df):.2%}")
    with col3:
        st.metric("Churn Rate", f"{sticky_churn:.2%}")
    
    # RSI distribution
    fig = go.Figure(data=[go.Histogram(x=df['RSI'], nbinsx=30, marker_color='#3498db')])
    fig.update_layout(title="Relationship Strength Index Distribution", xaxis_title="RSI Score")
    st.plotly_chart(fig, use_container_width=True)
    
    # Segment comparison
    segment_stats = df.groupby('Customer_Segment').agg({
        'Exited': ['count', 'mean'],
        'Balance': 'mean'
    })
    segment_stats.columns = ['Count', 'Churn_Rate', 'Avg_Balance']
    st.dataframe(segment_stats)
'''

# Save dashboard code
with open('../dashboard/app.py', 'w') as f:
    f.write(dashboard_code)

print("✓ Dashboard code created: dashboard/app.py")
print("\nTo run the dashboard, use:")
print("  streamlit run dashboard/app.py")

UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f4ca' in position 852: character maps to <undefined>

In [1]:
# Create requirements.txt
requirements_content = '''streamlit==1.28.0
pandas==2.0.0
numpy==1.24.0
plotly==5.18.0
scikit-learn==1.3.0
openpyxl==3.1.0
'''

with open('../requirements.txt', 'w') as f:
    f.write(requirements_content)

print("✓ requirements.txt created")

# Create README
readme_content = '''# Bank Customer Churn Analysis System

## Project Overview
This project analyzes why bank customers leave (churn) based on engagement, product utilization, and relationship strength.

## Project Structure
```
├── notebooks/
│   ├── Phase_1_Data_Understanding_Cleaning.ipynb
│   ├── Phase_2_Exploratory_Data_Analysis.ipynb
│   ├── Phase_3_Feature_Engineering.ipynb
│   ├── Phase_4_KPI_Business_Metrics.ipynb
│   └── Phase_5_Dashboard_Setup.ipynb
├── data/
│   ├── cleaned_data_with_features.csv
│   ├── data_with_engineered_features.csv
│   └── kpi_report.json
├── dashboard/
│   └── app.py
├── scripts/
└── requirements.txt
```

## Key Findings

### 1. Engagement Impact
- **Active customers churn at 13.28%**
- **Inactive customers churn at 36.60%**
- Active customers are **2.75x more likely to retain**

### 2. Product Utilization
- Single product: 27.73% churn
- Multi-product: 9.65% churn
- **More products = significantly lower churn**

### 3. Premium At-Risk
- 9.8% of customers are high-balance and inactive
- These customers churn at **69.28% rate**
- Represents major revenue risk

### 4. Sticky Customers
- 7.2% meet "sticky" criteria
- These customers churn at only **0.14%**
- Have 2+ products, active, long tenure, credit card

## Running the Analysis

### Step 1: Install Dependencies
```bash
pip install -r requirements.txt
```

### Step 2: Run Notebooks (In Order)
```bash
jupyter notebook notebooks/Phase_1_Data_Understanding_Cleaning.ipynb
jupyter notebook notebooks/Phase_2_Exploratory_Data_Analysis.ipynb
jupyter notebook notebooks/Phase_3_Feature_Engineering.ipynb
jupyter notebook notebooks/Phase_4_KPI_Business_Metrics.ipynb
jupyter notebook notebooks/Phase_5_Dashboard_Setup.ipynb
```

### Step 3: Launch Dashboard
```bash
streamlit run dashboard/app.py
```

## Features Created
- **Engagement Category**: 4 customer engagement levels
- **Product Depth Index**: Normalized product usage
- **Relationship Strength Index**: Composite loyalty score
- **Churn Risk Score**: Predictive risk indicator
- **Customer Value Score**: Revenue potential indicator
- **Customer Segment**: 4-tier customer classification

## KPI Metrics
1. **Engagement Retention Ratio**: 2.75x (active vs inactive)
2. **Multi-Product Penetration**: % of customers with 2+ products
3. **Premium Customer Churn**: Monitored for revenue protection
4. **Sticky Customer Rate**: Loyalty indicator
5. **At-Risk Balance**: Total revenue at churn risk

## Next Steps
1. Build churn prediction model (Logistic Regression, Random Forest, XGBoost)
2. Implement customer intervention recommendations
3. Automate dashboard updates with new data
4. A/B test retention initiatives

---

**Dashboard Last Updated**: May 2026
**Data Points**: 10,000 customers
**Analysis Scope**: European markets
'''

with open('../README.md', 'w') as f:
    f.write(readme_content)

print("✓ README.md created")
print("\n" + "="*80)
print("PROJECT SETUP COMPLETE!")
print("="*80)

✓ requirements.txt created


UnicodeEncodeError: 'charmap' codec can't encode characters in position 216-218: character maps to <undefined>